In [1]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD, State
from pgmpy.inference import VariableElimination
from pgmpy.sampling import BayesianModelSampling

c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\pgmpy\estimators\__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


In [2]:
model = DiscreteBayesianNetwork([
    ("heatwave", "overload"),
    ("heatwave", "powercut"),
    ("overload", "powercut"),
    ("powercut", "cancel")
])

In [3]:
# Probability of heatwave
cpd_heatwave = TabularCPD(
    variable="heatwave",
    variable_card=2,
    values=[[0.3], [0.7]],
    state_names={"heatwave": ["yes", "no"]}
)

# Probability of overload
cpd_overload = TabularCPD(
    variable="overload",
    variable_card=2,
    values=[
        [0.6, 0.1],
        [0.4, 0.9]
    ],
    evidence=["heatwave"],
    evidence_card=[2],
    state_names={
        "overload": ["yes", "no"],
        "heatwave": ["yes", "no"]
    }
)

# Probability of powercut
cpd_powercut = TabularCPD(
    variable="powercut",
    variable_card=2,
    values=[
        [0.7, 0.2, 0.5, 0.05],
        [0.3, 0.8, 0.5, 0.95]
    ],
    evidence=["heatwave", "overload"],
    evidence_card=[2, 2],
    state_names={
        "powercut": ["yes", "no"],
        "heatwave": ["yes", "no"],
        "overload": ["yes", "no"]
    }
)

# Probability of cancellation
cpd_cancel = TabularCPD(
    variable="cancel",
    variable_card=2,
    values=[
        [0.8, 0.1],
        [0.2, 0.9]
    ],
    evidence=["powercut"],
    evidence_card=[2],
    state_names={
        "cancel": ["yes", "no"],
        "powercut": ["yes", "no"]
    }
)

In [4]:
model.add_cpds(
    cpd_heatwave,
    cpd_overload,
    cpd_powercut,
    cpd_cancel
)

print(model.check_model())

True


In [5]:
inference = VariableElimination(model)

result = inference.query(
    variables=["heatwave"],
    evidence={"cancel": "yes"}
)

print(result)
print("Exact probability:", result.get_value(heatwave="yes"))

+---------------+-----------------+
| heatwave      |   phi(heatwave) |
+===============+=================+
| heatwave(yes) |          0.5367 |
+---------------+-----------------+
| heatwave(no)  |          0.4633 |
+---------------+-----------------+
Exact probability: 0.5366726296958855


The exact probability gives the precise chance of a heatwave occurring when the lab session is cancelled. The rejection sampling estimate uses 10,000 random samples to approximate this probability.

Both answers should be close to each other, but they may not be exactly the same because rejection sampling uses random samples.